In [18]:
from src.models.model import SegformerB5

# This downloads ~330 MB of weights from HuggingFace on first run
model = SegformerB5(
    # 14 matches the layout of the pre-baked GeoTIFFs: the 12 L2A spectral bands
    # (B10 is dropped by atmospheric correction) plus NDVI and NDWI as derived
    # channels. `src/download_region.py` produces the same layout. If you swap in
    # a different dataset (different band count or different derived layers), this
    # number, the normalisation statistics, and `num_channels` in the SegformerConfig
    # must all change together — otherwise the first patch-embedding layer
    # mis-shapes inputs.
    n_bands=14,
    logits=True,             # return raw logits (not probabilities)
    freeze_encoder=False,    # keep encoder trainable
    #type_labeler="CLCplus-Backbone",
)

Loading weights: 100%|██████████| 1156/1156 [00:00<00:00, 18768.13it/s]
[transformers] SemanticSegmentationSegformer LOAD REPORT from: nvidia/mit-b5
Key                                                     | Status     |                                                                                                  
--------------------------------------------------------+------------+--------------------------------------------------------------------------------------------------
classifier.bias                                         | UNEXPECTED |                                                                                                  
classifier.weight                                       | UNEXPECTED |                                                                                                  
decode_head.linear_fuse.weight                          | MISSING    |                                                                                                  
decode

In [11]:
def count_params(module):
    total = sum(p.numel() for p in module.parameters())
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    return total, trainable

total, trainable = count_params(model)
enc_total, enc_trainable = count_params(model.segformer)
dec_total, dec_trainable = count_params(model.decode_head)

print(f"{'Component':<20} {'Total params':>15} {'Trainable params':>18}")
print("-" * 55)
print(f"{'Encoder (MiT-B5)':<20} {enc_total:>15,} {enc_trainable:>18,}")
print(f"{'Decoder (MLP)':<20} {dec_total:>15,} {dec_trainable:>18,}")
print(f"{'Full model':<20} {total:>15,} {trainable:>18,}")

Component               Total params   Trainable params
-------------------------------------------------------
Encoder (MiT-B5)          81,477,504         81,477,504
Decoder (MLP)              3,159,564          3,159,564
Full model                84,637,068         84,637,068


In [12]:
model.freeze()

total, trainable = count_params(model)
print(f"After freezing encoder — trainable: {trainable:,} / {total:,}")

# Unfreeze for full fine-tuning
model.unfreeze()
total, trainable = count_params(model)
print(f"After unfreezing — trainable: {trainable:,} / {total:,}")

After freezing encoder — trainable: 3,159,564 / 84,637,068
After unfreezing — trainable: 84,637,068 / 84,637,068


In [21]:
# 1. Parameters per encoder stage.
# Deeper stages have more channels (C₁ < C₂ < C₃ < C₄), so even with fewer transformer
# blocks they end up holding most of the encoder's capacity. This matters during
# fine-tuning: freezing stages 3–4 alone already locks the bulk of the model's weights.
for i, stage in enumerate(model.segformer.encoder.block):
    n = sum(p.numel() for p in stage.parameters())
    print(f"Stage {i+1}: {n:,} parameters")

# 2. Decoder breakdown.
# The four projection layers are tiny linear maps (one per encoder stage); the real
# decoder cost lives in the linear_fuse + classifier. Seeing how light the head is
# explains why training the decoder alone is fast and resistant to overfitting.
for i, proj in enumerate(model.decode_head.linear_c):
    n = sum(p.numel() for p in proj.parameters())
    print(f"Decoder projection {i+1}: {n:,} parameters")

clf_params = sum(p.numel() for p in model.decode_head.classifier.parameters())
dec_total = sum(p.numel() for p in model.decode_head.parameters())
print(f"Classifier: {clf_params:,} / {dec_total:,} decoder params "
      f"({100*clf_params/dec_total:.1f}%)")

AttributeError: 'SegformerModel' object has no attribute 'encoder'

In [22]:
import torch

B, C, H, W = 2, 14, 512, 512   # batch size, channels, height, width
dummy_input = torch.randn(B, C, H, W)
dummy_labels = torch.randint(0, model.config.num_labels, (B, H, W))

print(f"Input  shape: {tuple(dummy_input.shape)}")
print(f"Labels shape: {tuple(dummy_labels.shape)}")

Input  shape: (2, 14, 512, 512)
Labels shape: (2, 512, 512)


In [23]:
model.eval()
with torch.no_grad():
    # Without labels → raw logits at H/4 × W/4
    logits = model(dummy_input)
    print(f"Logits shape (no labels):   {tuple(logits.shape)}")
    # Expected: (2, num_classes, 128, 128)  — quarter resolution

    # With labels → logits upsampled to label resolution
    upsampled = model(dummy_input, dummy_labels)
    print(f"Logits shape (with labels): {tuple(upsampled.shape)}")
    # Expected: (2, num_classes, 512, 512)

Logits shape (no labels):   (2, 12, 128, 128)
Logits shape (with labels): (2, 12, 512, 512)


In [24]:
outputs = model.segformer(
    dummy_input,
    output_hidden_states=True,
    return_dict=True,
)

for i, hs in enumerate(outputs.hidden_states):
    print(f"Stage {i+1} hidden state: {tuple(hs.shape)}")

Stage 1 hidden state: (2, 64, 128, 128)
Stage 2 hidden state: (2, 128, 64, 64)
Stage 3 hidden state: (2, 320, 32, 32)
Stage 4 hidden state: (2, 512, 16, 16)


In [31]:
import torch.nn.functional as F
# 1. Manual upsampling
logits_full = F.interpolate(
    model(dummy_input),                      # TODO: the raw logits tensor
    size=(512,512),                 # TODO: target (H, W)
    mode="bilinear",
    align_corners=False,
)
print(f"Manually upsampled: {tuple(logits_full.shape)}")

# 2. Predicted class map
probs = torch.softmax(logits, dim=1)   # TODO: softmax over class dimension
pred  = torch.argmax(probs, dim=1)   # TODO: argmax → (B, H, W)
print(f"Predicted map shape: {tuple(pred.shape)}")
print(f"Unique predicted classes: {pred.unique().tolist()}")

# 3. Spatial area ratios
#for i, hs in enumerate(outputs.hidden_states):
#    ratio =  pred.shape/(512,512)# TODO
#    print(f"Stage {i+1}: {ratio:.4f}")

Manually upsampled: (2, 12, 512, 512)
Predicted map shape: (2, 128, 128)
Unique predicted classes: [1, 3, 4, 5, 6, 7, 8, 9, 10, 11]


In [34]:
pip install pytorch_lightning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.2/852.2 kB 56.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 67.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pytorch_lightning]pytorch_lightning]
Note: you may need to restart the kernel to use updated packages.


In [35]:
from src.models.module import SegmentationModule
from torch import nn, optim

module = SegmentationModule(
    model=model,
    loss=nn.CrossEntropyLoss(ignore_index=255),
    optimizer=optim.AdamW,
    optimizer_params={"lr": 1e-3, "weight_decay": 1e-2},
    scheduler=optim.lr_scheduler.OneCycleLR,
    scheduler_params={},
    scheduler_interval="step",
)

In [36]:
from src.training.metrics import IOU, positive_rate

# Simulate model output and labels
B, num_classes, H, W = 2, 11, 512, 512
dummy_logits = torch.randn(B, num_classes, H, W)
dummy_labels = torch.randint(0, num_classes, (B, H, W))

iou_mean, iou_building = IOU(dummy_logits, dummy_labels, logits=True)
building_rate = positive_rate(dummy_logits, logits=True)

print(f"Mean IoU:      {iou_mean:.4f}")
print(f"Building IoU:  {iou_building:.4f}")
print(f"Building rate: {building_rate:.4f}  (fraction of pixels predicted as building)")

Mean IoU:      0.0474
Building IoU:  0.0472
Building rate: 0.0912  (fraction of pixels predicted as building)


In [37]:
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint

callbacks = [
    EarlyStopping(monitor="validation_loss", patience=5, mode="min"),
    ModelCheckpoint(
        monitor="validation_loss",
        mode="min",
        save_top_k=1,
        filename="best-model",
    ),
]

Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/opt/python/lib/python3.13/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /home/onyxia/work/funathon-project3/lightning_logs/version_0/checkpoints exists and is not empty.
Loading `train_dataloader` to estimate number of stepping batches.
/opt/python/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/opt/python/lib/python3.13/s

┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type                          ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ SemanticSegmentationSegformer │ 84.6 M │ eval  │     0 │
│ 1 │ loss  │ CrossEntropyLoss              │      0 │ train │     0 │
└───┴───────┴───────────────────────────────┴────────┴───────┴───────┘

Trainable params: 84.6 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 84.6 M                                                                                               
Total estimated model params size (MB): 338.548                                                                    
Modules in train mode: 1                                                                                           
Modules in eval mode: 1073                                                                                         
Total FLOPs: 0

/opt/python/lib/python3.13/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/opt/python/lib/python3.13/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 
'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=71` in the `DataLoader` to improve performance.

/opt/python/lib/python3.13/site-packages/pytorch_lightning/loops/fit_loop.py:538: Found 1073 module(s) in eval mode
at the start of training. This may lead to unexpected behavior during training. If this is intentional, you can 
ignore this warning.